# 가전 동작 상태 EDA — sub-state 라벨링 기준 수립

| 항목 | 내용 |
|------|------|
| 목적 | 켜짐 구간의 `active_power`, `power_factor`, `current` 분포를 시각화 + PLAID 상태 레퍼런스 오버레이로 임계값 초안 도출 |
| 대상 가전 | 에어컨, 선풍기, 전자레인지, 헤어드라이기 (PLAID 참고 가능 4종) |
| 데이터 | GCS `ax-nilm-data-dhwang0803-us` 30Hz 한국 데이터 |
| PLAID 통계 | 로컬에서 사전 계산 후 상수로 임베드 (110V → W/PF 직접 비교 가능, I_mean은 ÷2 보정) |
| 참고 | `docs/state_labeling_plan.md` |

In [ ]:
!pip install -q gcsfs pyarrow scipy

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCP 인증 완료')

In [ ]:
import gcsfs
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pyarrow as pa
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
from scipy.signal import find_peaks
import warnings, os, shutil, glob, subprocess
warnings.filterwarnings('ignore')

gcs = gcsfs.GCSFileSystem()

# 한글 폰트 설치
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)

# 폰트 파일 경로 직접 탐색 → addfont로 명시 등록 (캐시 우회)
ttf_candidates = glob.glob('/usr/share/fonts/**/*Nanum*Gothic*.ttf', recursive=True)
if ttf_candidates:
    fm.fontManager.addfont(ttf_candidates[0])
    prop = fm.FontProperties(fname=ttf_candidates[0])
    font_name = prop.get_name()
    plt.rcParams['font.family'] = font_name
    print(f'폰트 설정: {font_name}  ({ttf_candidates[0]})')
else:
    print('NanumGothic.ttf 파일 없음 — 설치 실패 가능성')
plt.rcParams['axes.unicode_minus'] = False

print('GCS 연결 완료')

In [ ]:
# 한글 폰트 렌더링 확인 — 아래 그래프 제목이 한글로 보이면 OK
fig, ax = plt.subplots(figsize=(4, 1))
ax.set_title('한글 폰트 테스트: 에어컨 선풍기 전자레인지')
ax.axis('off')
plt.tight_layout()
plt.show()

## PLAID 상태별 통계 (로컬 사전 계산 — 110V 기준)

- `W_mean`: 한국 데이터와 직접 비교 가능 (유효전력은 전압 무관)
- `PF_mean`: 직접 비교 가능 (부하 특성값)
- `I_mean`: 한국 데이터는 약 ÷2 수준 (220V → 같은 W에서 전류 절반)

> ⚠️ 에어컨 W_std가 매우 크고 highcool(93W) ≈ lowcool(90W) — W 단독 구분 불가, PF 차이(0.78 vs 0.76)로 보조 구분

In [ ]:
# PLAID 상태별 통계 (6170903/ 로컬 데이터 기반 사전 계산)
PLAID_STATS = {
    'Air Conditioner': {
        'highcool': {'W_mean': 93.05,  'PF_mean': 0.781, 'I_mean': 1.118, 'n': 39},
        'highfan':  {'W_mean': 35.76,  'PF_mean': 0.806, 'I_mean': 0.378, 'n': 34},
        'lowcool':  {'W_mean': 90.40,  'PF_mean': 0.755, 'I_mean': 1.180, 'n': 35},
        'lowfan':   {'W_mean': 29.90,  'PF_mean': 0.829, 'I_mean': 0.312, 'n': 34},
    },
    'Fan': {
        'high':   {'W_mean': 32.49, 'PF_mean': 0.770, 'I_mean': 0.357, 'n': 35},
        'medium': {'W_mean': 29.80, 'PF_mean': 0.776, 'I_mean': 0.315, 'n': 30},
        'low':    {'W_mean': 23.18, 'PF_mean': 0.757, 'I_mean': 0.249, 'n': 30},
    },
    'Microwave': {
        'high':    {'W_mean': 228.06, 'PF_mean': 0.559, 'I_mean': 3.475, 'n': 26},
        'medium':  {'W_mean': 210.15, 'PF_mean': 0.533, 'I_mean': 3.375, 'n': 44},
        'regular': {'W_mean': 220.60, 'PF_mean': 0.479, 'I_mean': 3.740, 'n': 20},
    },
    'Hairdryer': {
        'highhot':   {'W_mean': 580.62, 'PF_mean': 0.871, 'I_mean': 5.837, 'n': 20},
        'highwarm':  {'W_mean': 385.97, 'PF_mean': 0.878, 'I_mean': 3.806, 'n':  5},
        'highheat':  {'W_mean': 220.15, 'PF_mean': 0.790, 'I_mean': 2.447, 'n':  6},
        'highcool':  {'W_mean':  92.29, 'PF_mean': 0.859, 'I_mean': 0.919, 'n': 10},
        'mediumhot': {'W_mean': 262.43, 'PF_mean': 0.854, 'I_mean': 2.650, 'n':  5},
        'lowhot':    {'W_mean': 176.51, 'PF_mean': 0.828, 'I_mean': 1.837, 'n': 20},
        'lowwarm':   {'W_mean': 179.79, 'PF_mean': 0.809, 'I_mean': 1.901, 'n':  5},
        'lowheat':   {'W_mean': 125.81, 'PF_mean': 0.794, 'I_mean': 1.381, 'n':  6},
        'lowcool':   {'W_mean':  74.31, 'PF_mean': 0.897, 'I_mean': 0.704, 'n': 10},
        'noheat':    {'W_mean':  22.34, 'PF_mean': 0.770, 'I_mean': 0.251, 'n':  5},
    },
}

# 한국 가전명 → PLAID 가전명 매핑
KR_TO_PLAID = {
    '에어컨':     'Air Conditioner',
    '선풍기':     'Fan',
    '전자레인지': 'Microwave',
    '헤어드라이기': 'Hairdryer',
}

# 상태별 색상
STATE_COLORS = [
    '#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
    '#a65628','#f781bf','#999999','#66c2a5','#fc8d62',
]

print('PLAID 통계 로드 완료')

In [ ]:
import os
BUCKET_PREFIX = os.environ.get('GCS_NILM_DATA_PREFIX', '')
LABEL_PATH    = os.environ.get('GCS_NILM_LABEL_PATH', '')

TARGET_APPLIANCES = ['에어컨', '선풍기', '전자레인지', '헤어드라이기']

EDA_HOUSES = [
    'house_011', 'house_015', 'house_016', 'house_017',
    'house_033', 'house_039', 'house_049', 'house_054',
    'house_063', 'house_067'
]

## 1. 라벨 파일에서 가전별 가구/채널 목록 조회

In [ ]:
pa_fs = pa.fs.PyFileSystem(pa.fs.FSSpecHandler(gcs))
labels_df = pq.read_table(LABEL_PATH, filesystem=pa_fs).to_pandas()

print('라벨 컬럼:', labels_df.columns.tolist())
print('총 레코드:', len(labels_df))
labels_df.head(3)

In [ ]:
appliance_map = {}  # appliance_name → [(house_id, channel), ...]

for app in TARGET_APPLIANCES:
    rows = labels_df[
        labels_df['appliance_name'].str.contains(app, na=False) &
        labels_df['household_id'].isin(EDA_HOUSES)
    ][['household_id', 'channel']].drop_duplicates()
    appliance_map[app] = list(rows.itertuples(index=False, name=None))
    print(f'{app}: {len(appliance_map[app])}개 채널 → {appliance_map[app]}')

In [ ]:
def get_on_periods(house_id, channel):
    """라벨에서 (start_ts, end_ts) 켜짐 구간 리스트 반환."""
    rows = labels_df[
        (labels_df['household_id'] == house_id) &
        (labels_df['channel'] == channel)
    ]
    periods = []
    for _, row in rows.iterrows():
        s = pd.to_datetime(row.get('start_ts'))
        e = pd.to_datetime(row.get('end_ts'))
        if pd.notna(s) and pd.notna(e):
            periods.append((s, e))
    return periods


def extract_on_data(house_id, channel):
    """켜짐 구간 날짜 파티션만 GCS에서 읽어 on 구간 필터링.
    전체 채널 로드 대신 on 구간 날짜만 로드해 GCS I/O 최소화.
    """
    periods = get_on_periods(house_id, channel)
    if not periods:
        print(f'  [{house_id}/{channel}] 켜짐 구간 없음')
        return pd.DataFrame()

    # on 구간에 해당하는 날짜(YYYYMMDD) 집합
    on_dates = set()
    for s, e in periods:
        d = s.normalize()
        while d <= e.normalize():
            on_dates.add(int(d.strftime('%Y%m%d')))
            d += pd.Timedelta(days=1)

    # PyArrow predicate로 해당 날짜 파티션만 로드
    path = f'{BUCKET_PREFIX}/household_id={house_id}/channel={channel}'
    dataset = ds.dataset(path, filesystem=pa_fs, partitioning='hive')
    date_filter = ds.field('date').isin(list(on_dates))
    table = dataset.to_table(
        columns=['date_time', 'active_power', 'current', 'power_factor',
                 'apparent_power', 'reactive_power'],
        filter=date_filter
    )
    raw = table.to_pandas()
    raw['date_time'] = pd.to_datetime(raw['date_time'])
    raw = raw.sort_values('date_time').reset_index(drop=True)

    # 정확한 on 구간만 필터
    mask = pd.Series(False, index=raw.index)
    for s, e in periods:
        mask |= (raw['date_time'] >= s) & (raw['date_time'] <= e)

    on_df = raw[mask].copy()
    on_df['house_channel'] = f'{house_id}/{channel}'
    print(f'  [{house_id}/{channel}] 로드 날짜: {len(on_dates)}일 | 켜짐 샘플: {len(on_df):,}개 ({len(periods)}구간)')
    return on_df

print('함수 정의 완료')

## 2. 켜짐 구간 raw 데이터 추출

In [ ]:
def load_raw_channel(house_id, channel):
    path = f'{BUCKET_PREFIX}/household_id={house_id}/channel={channel}'
    dataset = ds.dataset(path, filesystem=pa_fs, partitioning='hive')
    df = dataset.to_table(
        columns=['date_time', 'active_power', 'current', 'power_factor', 'apparent_power', 'reactive_power']
    ).to_pandas()
    # timezone-naive UTC로 통일 (KST naive와 UTC naive 혼재 방지)
    df['date_time'] = pd.to_datetime(df['date_time']).dt.tz_localize(None)
    return df.sort_values('date_time').reset_index(drop=True)


def get_on_periods(house_id, channel):
    rows = labels_df[
        (labels_df['household_id'] == house_id) &
        (labels_df['channel'] == channel)
    ]
    periods = []
    for _, row in rows.iterrows():
        s = pd.to_datetime(row.get('start_ts'))
        e = pd.to_datetime(row.get('end_ts'))
        if pd.notna(s) and pd.notna(e):
            # tz-aware면 UTC로 변환 후 naive로; naive면 그대로
            s = s.tz_convert('UTC').tz_localize(None) if s.tzinfo else s
            e = e.tz_convert('UTC').tz_localize(None) if e.tzinfo else e
            periods.append((s, e))
    return periods


def extract_on_data(house_id, channel):
    raw = load_raw_channel(house_id, channel)
    periods = get_on_periods(house_id, channel)
    if not periods:
        return pd.DataFrame()
    mask = pd.Series(False, index=raw.index)
    for s, e in periods:
        mask |= (raw['date_time'] >= s) & (raw['date_time'] <= e)
    on_df = raw[mask].copy()
    on_df['house_channel'] = f'{house_id}/{channel}'
    n_matched = mask.sum()
    print(f'  [{house_id}/{channel}] 켜짐 샘플: {n_matched:,}개 ({len(periods)}구간) | mean W={on_df["active_power"].mean():.1f}')
    return on_df

print('함수 정의 완료')

## 3. 가전별 켜짐 구간 데이터 수집

In [ ]:
on_data = {}

for app, channels in appliance_map.items():
    if not channels:
        print(f'[{app}] 데이터 없음, 스킵')
        continue
    print(f'\n[{app}] 수집 중...')
    frames = [extract_on_data(h, ch) for h, ch in channels]
    frames = [f for f in frames if not f.empty]
    if frames:
        on_data[app] = pd.concat(frames, ignore_index=True)
        print(f'  → 합계: {len(on_data[app]):,}개 샘플')

print('\n수집 완료')

In [ ]:
# 채널 품질 진단 — 가전별 최소 기대 전력 미달 채널 식별
POWER_MIN_THRESH = {
    '에어컨':      25.0,   # lowfan(30W)의 80%
    '선풍기':      15.0,   # Fan low(23W)의 65%
    '전자레인지': 100.0,   # Microwave 최저(210W)의 50%
    '헤어드라이기': 15.0,  # noheat(22W)의 65%
}

bad_channels = {}
for app, df in on_data.items():
    thresh = POWER_MIN_THRESH.get(app, 0)
    per_ch = (
        df.groupby('house_channel')['active_power']
        .agg(mean='mean', median='median', n='count')
        .reset_index()
    )
    per_ch['ok'] = per_ch['mean'] >= thresh
    bad = per_ch[~per_ch['ok']]
    bad_channels[app] = bad['house_channel'].tolist()

    print(f'\n[{app}] 채널별 mean active_power (임계값 ≥{thresh}W)')
    print(per_ch.to_string(index=False))
    if not bad.empty:
        print(f'  ⚠️  품질 미달 채널: {bad["house_channel"].tolist()}')

## 4. 히스토그램 + PLAID 상태 레퍼런스 오버레이

- **세로 실선**: PLAID 각 상태의 W_mean / PF_mean (한국 데이터와 직접 비교)
- 한국 데이터 군집이 PLAID 레퍼런스 선과 겹치는 위치 → 해당 상태로 라벨링 후보

In [ ]:
FEATURE_COLS = [
    ('active_power', '유효전력 W',  'W_mean'),
    ('power_factor', '역률 PF',     'PF_mean'),
    ('current',      '전류 I (A)',  'I_mean'),   # PLAID I_mean ÷2 보정 적용
]

for app, df in on_data.items():
    plaid_key = KR_TO_PLAID.get(app)
    plaid = PLAID_STATS.get(plaid_key, {})
    states = list(plaid.keys())

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    fig.suptitle(
        f'{app} — 켜짐 구간 분포  |  세로선: PLAID 상태 레퍼런스 (n={len(df):,})',
        fontsize=12, fontweight='bold'
    )

    for ax, (col, label, plaid_field) in zip(axes, FEATURE_COLS):
        if col not in df.columns:
            ax.set_title(f'{label} (컬럼 없음)')
            continue

        vals = df[col].dropna()
        vals = vals[vals <= vals.quantile(0.99)]

        ax.hist(vals, bins=80, color='steelblue', alpha=0.6, edgecolor='white', linewidth=0.3)
        ax.axvline(vals.mean(),   color='black', linestyle='--', linewidth=1.2,
                   label=f'KR mean={vals.mean():.1f}')

        # PLAID 상태별 레퍼런스 라인
        for i, (state, stats) in enumerate(plaid.items()):
            ref_val = stats.get(plaid_field, None)
            if ref_val is None:
                continue
            # 전류는 220V 보정 (÷2)
            if plaid_field == 'I_mean':
                ref_val = ref_val / 2
            color = STATE_COLORS[i % len(STATE_COLORS)]
            ax.axvline(ref_val, color=color, linestyle='-', linewidth=1.5, alpha=0.85,
                       label=f'{state} ({ref_val:.1f})')

        ax.set_title(label, fontsize=10)
        ax.set_xlabel(label)
        ax.set_ylabel('샘플 수')
        ax.legend(fontsize=6.5, loc='upper right')
        ax.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.show()
    print()

In [ ]:
SKIP_APPS = {'전자레인지'}  # on/off 단순 → 시계열 생략

plot_apps = [a for a in TARGET_APPLIANCES if a in on_data and a not in SKIP_APPS]
fig, axes = plt.subplots(len(plot_apps), 1,
                         figsize=(18, 3.5 * len(plot_apps)), sharex=False)
if len(plot_apps) == 1:
    axes = [axes]

for ax, app in zip(axes, plot_apps):
    plaid_key = KR_TO_PLAID.get(app)
    plaid = PLAID_STATS.get(plaid_key, {})
    df = on_data[app].copy()

    # mean W 가장 높은 채널 선택 (품질 기준)
    best_ch = df.groupby('house_channel')['active_power'].mean().idxmax()
    df_ch = df[df['house_channel'] == best_ch].set_index('date_time').sort_index()
    df_plot = df_ch['active_power'].iloc[:24 * 30 * 3600]

    ax.plot(df_plot.index, df_plot.values, linewidth=0.6, color='steelblue', alpha=0.8)

    for i, (state, stats) in enumerate(plaid.items()):
        color = STATE_COLORS[i % len(STATE_COLORS)]
        ax.axhline(stats['W_mean'], color=color, linestyle='--', linewidth=1.0,
                   alpha=0.7, label=f'{state} ({stats["W_mean"]:.0f}W)')

    mean_w = df_ch['active_power'].mean()
    ax.set_title(f'{app} — {best_ch}  (mean={mean_w:.1f}W)  |  수평선: PLAID W_mean', fontsize=10)
    ax.set_ylabel('W')
    ax.legend(fontsize=6.5, loc='upper right', ncol=3)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 임계값 초안 자동 추정 (valley detection)

히스토그램의 골짜기 위치 + PLAID 레퍼런스를 대조해 최종 임계값을 결정한다.

In [ ]:
def suggest_thresholds(app, col='active_power', n_bins=100):
    if app not in on_data:
        print(f'{app} 데이터 없음'); return

    plaid_key = KR_TO_PLAID.get(app)
    plaid = PLAID_STATS.get(plaid_key, {})

    vals = on_data[app][col].dropna()
    vals = vals[vals <= vals.quantile(0.99)]
    counts, bin_edges = np.histogram(vals, bins=n_bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    peaks, _ = find_peaks(counts, prominence=counts.max() * 0.1, distance=5)

    print(f'\n[{app}] {col} — 군집 중심 후보:')
    for p in peaks:
        print(f'  KR 피크: {bin_centers[p]:.1f}')

    print(f'  PLAID 레퍼런스:')
    plaid_field = 'W_mean' if col == 'active_power' else ('PF_mean' if col == 'power_factor' else 'I_mean')
    for state, stats in plaid.items():
        ref = stats[plaid_field]
        if plaid_field == 'I_mean': ref /= 2  # 220V 보정
        print(f'    {state}: {ref:.2f}')

    if len(peaks) >= 2:
        print(f'  골짜기 임계값 초안:')
        for i in range(len(peaks) - 1):
            valley_idx = peaks[i] + np.argmin(counts[peaks[i]:peaks[i+1]])
            print(f'    구간 {i+1}|{i+2}: {bin_centers[valley_idx]:.1f}')

for app in TARGET_APPLIANCES:
    suggest_thresholds(app, col='active_power')
    suggest_thresholds(app, col='power_factor')

## 7. 선풍기 전 채널 시계열 — 속도 전환 패턴 확인

In [ ]:
TARGET_CHANNEL_APP = '선풍기'
PLAID_FAN = PLAID_STATS['Fan']

if TARGET_CHANNEL_APP in on_data:
    df = on_data[TARGET_CHANNEL_APP]
    channels = sorted(df['house_channel'].unique())
    fig, axes = plt.subplots(len(channels), 1,
                             figsize=(18, 2.8 * len(channels)), sharex=False)
    if len(channels) == 1:
        axes = [axes]

    for ax, ch in zip(axes, channels):
        df_ch = df[df['house_channel'] == ch].set_index('date_time').sort_index()
        # 5일치만 표시 (전환 패턴 확인용)
        df_plot = df_ch['active_power'].iloc[:5 * 24 * 30 * 3600]
        mean_w = df_ch['active_power'].mean()

        ax.plot(df_plot.index, df_plot.values, linewidth=0.5, color='steelblue', alpha=0.8)
        # 골짜기 임계값
        ax.axhline(22.6, color='orange', linestyle='--', linewidth=1.0, label='low|medium (22.6W)')
        ax.axhline(32.9, color='red',    linestyle='--', linewidth=1.0, label='medium|high (32.9W)')
        for i, (state, stats) in enumerate(PLAID_FAN.items()):
            color = STATE_COLORS[i % len(STATE_COLORS)]
            ax.axhline(stats['W_mean'], color=color, linestyle=':', linewidth=0.8, alpha=0.6,
                       label=f'PLAID {state} ({stats["W_mean"]:.0f}W)')
        ax.set_title(f'{ch}  (mean={mean_w:.1f}W)', fontsize=9)
        ax.set_ylabel('W')
        ax.set_ylim(-2, 55)
        ax.legend(fontsize=6, loc='upper right', ncol=3)
        ax.grid(True, alpha=0.3)

    plt.suptitle('선풍기 — 전 채널 시계열 (5일, 골짜기 임계값 오버레이)', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 8. 전자레인지 전 채널 시계열 — 출력 설정 vs 모델 차이 확인

In [ ]:
if '전자레인지' in on_data:
    df = on_data['전자레인지']
    channels = sorted(df['house_channel'].unique())
    fig, axes = plt.subplots(len(channels), 1,
                             figsize=(18, 2.8 * len(channels)), sharex=False)
    if len(channels) == 1:
        axes = [axes]

    for ax, ch in zip(axes, channels):
        df_ch = df[df['house_channel'] == ch].set_index('date_time').sort_index()
        df_plot = df_ch['active_power']
        mean_w = df_ch['active_power'].mean()

        ax.plot(df_plot.index, df_plot.values, linewidth=0.5, color='steelblue', alpha=0.8)
        ax.axhline(692,  color='orange', linestyle='--', linewidth=1.0, label='standby|low (692W)')
        ax.axhline(996,  color='red',    linestyle='--', linewidth=1.0, label='low|high (996W)')
        ax.set_title(f'{ch}  (mean={mean_w:.0f}W)', fontsize=9)
        ax.set_ylabel('W')
        ax.legend(fontsize=7, loc='upper right')
        ax.grid(True, alpha=0.3)

    plt.suptitle('전자레인지 — 전 채널 시계열 (골짜기 임계값 오버레이)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 9. 전체 가전 골짜기 탐지

라벨 파일에 존재하는 모든 가전의  분포를 히스토그램으로 시각화하고 골짜기(valley) 임계값 초안을 도출한다.
가구 필터 없이 전체 라벨 데이터를 사용해 일반화된 분포를 확보한다.

In [ ]:
# 라벨 파일에서 전체 가전 목록 조회 (GCS 접근 없음)
all_apps = sorted(labels_df["appliance_name"].dropna().unique().tolist())
print(f"라벨에 존재하는 가전: {len(all_apps)}종")

# 가전별 가구/채널 매핑 (전체 가구, EDA_HOUSES 필터 없음)
appliance_map_all = {}
for app in all_apps:
    rows = labels_df[
        labels_df["appliance_name"] == app
    ][["household_id", "channel"]].drop_duplicates()
    appliance_map_all[app] = list(rows.itertuples(index=False, name=None))
    print(f"  {app}: {len(appliance_map_all[app])}개 채널")

In [ ]:
# 히스토그램 EDA용 — 채널 전체를 1회 로드 후 W > MIN_W 필터 (on-구간 정밀 필터 불필요)
# 기존 extract_on_data는 구간마다 GCS 왕복 → 냉장고 2840구간이면 2840번 → 수 시간 소요
MIN_W = 5.0           # off 상태 제외 임계값
MAX_ROWS_PER_APP = 1_000_000  # 히스토그램은 1M 샘플로 충분

on_data_all = {}
for app, pairs in appliance_map_all.items():
    valid_pairs = [(h, c) for h, c in pairs if h in EDA_HOUSES]
    if not valid_pairs:
        print(f"[{app}] EDA_HOUSES 내 라벨 없음 — 스킵")
        continue

    frames = []
    for house_id, ch in valid_pairs:
        try:
            df = load_raw_channel(house_id, ch)
            df = df[df["active_power"] > MIN_W][["active_power", "power_factor", "current"]]
            if not df.empty:
                frames.append(df)
        except Exception as e:
            print(f"  [{app}] {house_id}/{ch} 실패: {e}")

    if frames:
        combined = pd.concat(frames, ignore_index=True)
        if len(combined) > MAX_ROWS_PER_APP:
            combined = combined.sample(MAX_ROWS_PER_APP, random_state=42)
        on_data_all[app] = combined
        print(f"[{app}] {len(frames)}채널, {len(on_data_all[app]):,}행")
    else:
        print(f"[{app}] 데이터 없음 — 스킵")

In [ ]:
import math
apps = sorted(on_data_all.keys())
ncols = 4
nrows = math.ceil(len(apps) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 3))
axes = axes.flatten()

valley_results = {}

for idx, app in enumerate(apps):
    ax = axes[idx]
    vals = on_data_all[app]["active_power"].dropna()
    vals = vals[vals > 0]
    vals = vals[vals <= vals.quantile(0.99)]

    counts, bin_edges = np.histogram(vals, bins=100)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    peaks, _ = find_peaks(counts, prominence=counts.max() * 0.1, distance=5)
    valleys = []
    for i in range(len(peaks) - 1):
        vi = peaks[i] + np.argmin(counts[peaks[i]:peaks[i+1]])
        valleys.append(round(bin_centers[vi], 1))

    bw = bin_edges[1] - bin_edges[0]
    ax.bar(bin_centers, counts, width=bw, alpha=0.6)
    for v in valleys:
        ax.axvline(v, color="red", linestyle="--", linewidth=1.2, label=f"{v}W")
    for p in peaks:
        ax.axvline(bin_centers[p], color="green", linestyle=":", linewidth=1, alpha=0.7)
    ax.set_title(app, fontsize=10)
    ax.set_xlabel("W", fontsize=8)
    if valleys:
        ax.legend(fontsize=7, loc="upper right")

    valley_results[app] = {
        "n_samples": len(vals),
        "peaks": [round(bin_centers[p], 1) for p in peaks],
        "valleys": valleys
    }

for idx in range(len(apps), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
os.makedirs("eda_results", exist_ok=True)
plt.savefig("eda_results/전체가전_히스토그램.png", dpi=100)
plt.show()

# 요약 테이블
print("=== 골짜기 탐지 결과 요약 ===")
print(f"{'가전':<15} {'샘플수':>10} {'피크수':>6}  {'피크(W)':<40} 골짜기(W)")
print("-" * 90)
for app in apps:
    r = valley_results[app]
    print(f"{app:<15} {r['n_samples']:>10,} {len(r['peaks']):>6}  {str(r['peaks']):<40} {r['valleys']}")

## 10. 전체 가전 K-Means 클러스터링

K=2~6 범위에서 실루엣 점수로 최적 K를 선택하고, 골짜기 탐지 결과와 비교한다.
이 메모리에 있어야 함 (cell 28 실행 후).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

K_MIN, K_MAX = 2, 6
SKIP_APPS = {"메인 분전반", "무선공유기/셋톱박스"}  # 단일상태 또는 제외 대상

kmeans_results = {}  # app → {best_k, centers, labels, scores}

for app, df in on_data_all.items():
    if app in SKIP_APPS:
        continue
    X = df["active_power"].dropna().values.reshape(-1, 1)
    if len(X) < K_MAX * 10:
        print(f"[{app}] 샘플 부족 — 스킵")
        continue

    best_k, best_score, best_model = 2, -1, None
    scores = {}
    for k in range(K_MIN, K_MAX + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        score = silhouette_score(X, labels, sample_size=min(50000, len(X)), random_state=42)
        scores[k] = round(score, 4)
        if score > best_score:
            best_score, best_k, best_model = score, k, km

    centers = sorted(best_model.cluster_centers_.flatten().tolist())
    kmeans_results[app] = {"best_k": best_k, "centers": [round(c, 1) for c in centers], "scores": scores}
    print(f"[{app}] 최적 K={best_k} (실루엣={best_score:.4f}) | 중심: {[round(c,1) for c in centers]}")
    print(f"  실루엣 by K: {scores}")

In [ ]:
# valley vs K-Means 비교 그리드
import math
apps = sorted(kmeans_results.keys())
ncols = 4
nrows = math.ceil(len(apps) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 3))
axes = axes.flatten()

for idx, app in enumerate(apps):
    ax = axes[idx]
    vals = on_data_all[app]["active_power"].dropna()
    vals = vals[vals > 0]
    vals = vals[vals <= vals.quantile(0.99)]

    counts, bin_edges = np.histogram(vals, bins=100)
    bw = bin_edges[1] - bin_edges[0]
    ax.bar((bin_edges[:-1] + bin_edges[1:]) / 2, counts, width=bw, alpha=0.5, color="steelblue")

    # valley 골짜기 (빨간 점선)
    if app in valley_results:
        for v in valley_results[app]["valleys"]:
            ax.axvline(v, color="red", linestyle="--", linewidth=1.2, alpha=0.8)

    # K-Means 중심 (초록 실선)
    res = kmeans_results[app]
    for c in res["centers"]:
        ax.axvline(c, color="green", linestyle="-", linewidth=1.5, alpha=0.8)

    ax.set_title(f"{app}  K={res['best_k']}(sil={max(res['scores'].values()):.3f})", fontsize=9)
    ax.set_xlabel("W", fontsize=8)

for idx in range(len(apps), len(axes)):
    axes[idx].set_visible(False)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="red", linestyle="--", label="valley 골짜기"),
    Line2D([0], [0], color="green", linestyle="-", label="K-Means 중심"),
]
fig.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
os.makedirs("eda_results", exist_ok=True)
plt.savefig("eda_results/전체가전_클러스터링비교.png", dpi=100)
plt.show()

# 요약 테이블
print("=== valley vs K-Means 비교 ===")
print(f"{'가전':<18} {'valley피크':>8} {'KM최적K':>8}  {'KM중심(W)':<50} {'실루엣점수'}")
print("-" * 100)
for app in apps:
    vp = len(valley_results.get(app, {}).get("peaks", []))
    res = kmeans_results[app]
    print(f"{app:<18} {vp:>8} {res['best_k']:>8}  {str(res['centers']):<50} {res['scores']}")

## 11. 채널 오염 의심 가전 재클러스터링

W>5W 필터가 같은 채널의 다른 가전 데이터를 포함해 이상 중심값이 발생한 가전.
가전별 도메인 지식 기반 상한값(MAX_W)으로 오염 구간을 제거 후 K-Means 재실행.

In [ ]:
# 오염 의심 3종만 extract_on_data로 정밀 로드 (켜짐 구간만)
RECHECK_NAMES = ['에어컨', '김치 냉장고', '제습기']

on_data_recheck = {}
for app in RECHECK_NAMES:
    pairs = [(h, c) for h, c in appliance_map_all.get(app, [])
             if h in EDA_HOUSES]
    if not pairs:
        print(f'[{app}] EDA_HOUSES 내 라벨 없음 — 스킵')
        continue

    frames = []
    for house_id, ch in pairs:
        try:
            df = extract_on_data(house_id, ch)
            if not df.empty:
                frames.append(df)
        except Exception as e:
            print(f'  [{app}] {house_id}/{ch} 실패: {e}')

    if frames:
        on_data_recheck[app] = pd.concat(frames, ignore_index=True)
        print(f'[{app}] {len(frames)}채널, {len(on_data_recheck[app]):,}행')
    else:
        print(f'[{app}] 데이터 없음')

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RECHECK_APPS = {
    '에어컨':     50,
    '김치 냉장고': 200,
    '제습기':     500,
}
K_MIN, K_MAX = 2, 6

recluster_results = {}

for app, max_w in RECHECK_APPS.items():
    if app not in on_data_recheck:
        print(f'[{app}] on_data_recheck에 없음 — 스킵')
        continue

    raw = on_data_recheck[app]['active_power'].dropna()
    clean = raw[(raw > 0) & (raw <= max_w)]
    removed = len(raw) - len(clean)
    print(f'[{app}] 상한 {max_w}W 적용: {len(raw):,} → {len(clean):,}행 ({removed:,}행 제거)')

    if len(clean) < K_MAX * 10:
        print(f'  샘플 부족 — 스킵')
        continue

    X = clean.values.reshape(-1, 1)
    best_k, best_score, best_model = 2, -1, None
    scores = {}
    for k in range(K_MIN, K_MAX + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        score = silhouette_score(X, labels, sample_size=min(50000, len(X)), random_state=42)
        scores[k] = round(score, 4)
        if score > best_score:
            best_score, best_k, best_model = score, k, km

    centers = sorted(best_model.cluster_centers_.flatten().tolist())
    recluster_results[app] = {'best_k': best_k, 'centers': [round(c,1) for c in centers], 'scores': scores}
    print(f'  → 최적 K={best_k} (실루엣={best_score:.4f}) | 중심: {[round(c,1) for c in centers]}')
    print(f'  실루엣 by K: {scores}')

In [ ]:
# 정제 전후 히스토그램 비교
fig, axes = plt.subplots(len(RECHECK_APPS), 2, figsize=(12, len(RECHECK_APPS) * 3))

for row, (app, max_w) in enumerate(RECHECK_APPS.items()):
    if app not in on_data_recheck:
        continue
    raw = on_data_recheck[app]['active_power'].dropna()
    clean = raw[(raw > 0) & (raw <= max_w)]

    # 정제 전
    ax0 = axes[row, 0]
    v = raw[raw <= raw.quantile(0.99)]
    ax0.hist(v, bins=100, alpha=0.7, color='steelblue')
    ax0.set_title(f'{app} — 정제 전 (W≤{raw.quantile(0.99):.0f})', fontsize=9)
    ax0.set_xlabel('W')

    # 정제 후 + K-Means 중심
    ax1 = axes[row, 1]
    ax1.hist(clean, bins=100, alpha=0.7, color='tomato')
    if app in recluster_results:
        for c in recluster_results[app]['centers']:
            ax1.axvline(c, color='green', linewidth=1.8, linestyle='-', label=f'{c:.0f}W')
        ax1.legend(fontsize=8)
    ax1.set_title(f'{app} — 정제 후 (W≤{max_w}) | K={recluster_results.get(app, {}).get("best_k","?")}', fontsize=9)
    ax1.set_xlabel('W')

plt.tight_layout()
os.makedirs('eda_results', exist_ok=True)
plt.savefig('eda_results/오염가전_재클러스터링.png', dpi=100)
plt.show()

## 12. 세탁기 단일 사이클 시계열 — K=2 충분한지 확인

세탁기는 하나의 켜짐 구간 안에서 전력이 자동으로 변하므로,
히스토그램이 아닌 **시계열**로 사이클 단계를 확인해야 한다.

In [ ]:
# 세탁기 라벨 있는 EDA_HOUSES 채널 조회
washer_pairs = [(h, c) for h, c in appliance_map_all.get('세탁기', [])
                if h in EDA_HOUSES]
print(f'세탁기 채널: {washer_pairs}')

# 첫 번째 가구/채널에서 온-구간 목록 확인
house_id, ch = washer_pairs[0]
periods = get_on_periods(house_id, ch)
print(f'[{house_id}/{ch}] 켜짐 구간 수: {len(periods)}')

# 구간 길이 순으로 정렬 → 가장 긴 구간(완전한 세탁 사이클) 선택
periods_sorted = sorted(periods, key=lambda x: x[1] - x[0], reverse=True)
for i, (s, e) in enumerate(periods_sorted[:5]):
    print(f'  구간 {i+1}: {s} ~ {e} ({(e-s).total_seconds()/60:.1f}분)')

# 가장 긴 구간 로드
target_s, target_e = periods_sorted[0]
df_raw = load_raw_channel(house_id, ch)
df_cycle = df_raw[
    (df_raw['date_time'] >= target_s) &
    (df_raw['date_time'] <= target_e)
].copy()
print(f'선택 구간: {target_s} ~ {target_e} ({len(df_cycle):,}행)')

# 시계열 플롯
fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

axes[0].plot(df_cycle['date_time'], df_cycle['active_power'], color='steelblue', linewidth=0.6)
axes[0].axhline(1001.5, color='red', linestyle='--', linewidth=1.2, label='K=2 경계 (1001.5W)')
axes[0].set_ylabel('active_power (W)')
axes[0].set_title(f'세탁기 단일 사이클 시계열 — {house_id}/{ch}')
axes[0].legend(fontsize=9)

axes[1].plot(df_cycle['date_time'], df_cycle['current'], color='orange', linewidth=0.6)
axes[1].set_ylabel('current (A)')
axes[1].set_xlabel('시간')

plt.tight_layout()
os.makedi
rs('eda_results', exist_ok=True)
plt.savefig('eda_results/세탁기_사이클_시계열.png', dpi=100)
plt.show()
print(f'구간 내 W 통계: min={df_cycle["active_power"].min():.1f}, max={df_cycle["active_power"].max():.1f}, mean={df_cycle["active_power"].mean():.1f}')

## 13. 세탁기 재클러스터링

시계열 확인 결과 실제 max ~585W로, on_data_all 기반 K=2(94/1908W)는 오염된 결과.
extract_on_data로 켜짐 구간만 정밀 로드 후 ≤700W 필터 적용해 재클러스터링.

In [ ]:
# 세탁기 켜짐 구간만 정밀 로드 (extract_on_data)
washer_pairs = [(h, c) for h, c in appliance_map_all.get('세탁기', [])
                if h in EDA_HOUSES]

frames = []
for house_id, ch in washer_pairs:
    try:
        df = extract_on_data(house_id, ch)
        if not df.empty:
            frames.append(df)
            print(f'[{house_id}/{ch}] {len(df):,}행, max={df["active_power"].max():.1f}W')
    except Exception as e:
        print(f'[{house_id}/{ch}] 실패: {e}')

washer_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f'전체: {len(washer_df):,}행, max={washer_df["active_power"].max():.1f}W, mean={washer_df["active_power"].mean():.1f}W')

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

WASHER_MAX_W = 700
K_MIN, K_MAX = 2, 5

raw = washer_df['active_power'].dropna()
clean = raw[(raw > 0) & (raw <= WASHER_MAX_W)]
print(f'정제 후: {len(clean):,}행 (제거: {len(raw)-len(clean):,}행)')

X = clean.values.reshape(-1, 1)
best_k, best_score, best_model = 2, -1, None
scores = {}
for k in range(K_MIN, K_MAX + 1):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels, sample_size=min(50000, len(X)), random_state=42)
    scores[k] = round(score, 4)
    if score > best_score:
        best_score, best_k, best_model = score, k, km

centers = sorted(best_model.cluster_centers_.flatten().tolist())
boundaries = [(centers[i]+centers[i+1])/2 for i in range(len(centers)-1)]
print(f'최적 K={best_k} (실루엣={best_score:.4f})')
print(f'중심: {[round(c,1) for c in centers]}')
print(f'경계: {[round(b,1) for b in boundaries]}')
print(f'실루엣 by K: {scores}')

# 히스토그램 + K-Means 중심
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(clean, bins=100, alpha=0.6, color='steelblue')
for c in centers:
    ax.axvline(c, color='green', linewidth=2, label=f'중심 {c:.0f}W')
for b in boundaries:
    ax.axvline(b, color='red', linestyle='--', linewidth=1.5, label=f'경계 {b:.0f}W')
ax.set_title(f'세탁기 재클러스터링 (≤{WASHER_MAX_W}W) | K={best_k} sil={best_score:.3f}')
ax.set_xlabel('W')
ax.legend(fontsize=8)
plt.tight_layout()
os.makedirs('eda_results', exist_ok=True)
plt.savefig('eda_results/세탁기_재클러스터링.png', dpi=100)
plt.show()